# Week 8 Lab: Training GRPO with GSM8K

**Lab Goal:** Train a model to reason through math problems using Group Relative Policy Optimization (GRPO) and understand the shift from Supervised Fine-Tuning (SFT) to Reinforcement Learning (RL).

---

## Theory: How GRPO Works

**Group Relative Policy Optimization (GRPO)** was introduced by DeepSeek (DeepSeekMath, 2024) as a more efficient alternative to PPO (Proximal Policy Optimization).

### 1. The Core Innovation: No Critic Network
In standard RLHF (PPO), we need two models:
1.  **Policy Model:** The LLM we are training.
2.  **Value Model (Critic):** A second model that predicts the "expected reward" for a state.

The Critic is often as large as the Policy, doubling the VRAM requirements. **GRPO eliminates the Critic.** Instead, it uses **group statistics** to estimate the baseline.

### 2. The Mechanism
For every input prompt $q$, GRPO does the following:
1.  **Group Sampling:** Generates a group of $G$ different outputs $\{o_1, o_2, ..., o_G\}$ from the current policy.
2.  **Reward Calculation:** Scores each output using a reward function $r(o_i)$.
3.  **Advantage Estimation:** Instead of comparing an output to a Critic's prediction, it compares the output to the **mean reward of the group**:
    $$A_i = \frac{r_i - \text{mean}(r)}{\text{std}(r)}$$
    *   If an output is better than the group average, the advantage is positive (increase probability).
    *   If an output is worse than average, the advantage is negative (decrease probability).
4.  **Optimization:** Updates the model to maximize the objective while adding a **KL Divergence** penalty to ensure the model doesn't drift too far from the original "Reference Model" (preventing reward hacking).

---
## Part 1 – Environment Setup

In [1]:
# Install required libraries
!pip install -U transformers datasets accelerate peft trl bitsandbytes

In [2]:
import torch
import re
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import GRPOTrainer, GRPOConfig
from peft import LoraConfig
from tqdm import tqdm

---
## Part 2 – Load GSM8K

In [3]:
# 1. Load GSM8K Dataset
dataset = load_dataset("openai/gsm8k", "main")

def extract_hash_answer(text):
    if "####" not in text:
        return None
    return text.split("####")[-1].strip().replace(",", "")

def format_data(example):
    return {
        "prompt": [
            {"role": "system", "content": "You are a helpful assistant that reasons step-by-step. Provide your thought process inside <thought> tags and the final numeric answer inside <answer> tags."},
            {"role": "user", "content": example["question"]}
        ],
        "answer": extract_hash_answer(example["answer"])
    }

train_dataset = dataset["train"].map(format_data)
test_dataset = dataset["test"].map(format_data)

# Helper to extract numbers from model output
def extract_numeric_answer(text):
    # Search for content inside <answer> tags first
    match = re.search(r"<answer>(.*?)</answer>", text, re.DOTALL)
    if match:
        extracted = match.group(1).strip()
    else:
        # Fallback for baseline/CoT: look for "The answer is X"
        ans_match = re.search(r"[Tt]he answer is\s*([\d\.,]+)", text)
        if ans_match:
            extracted = ans_match.group(1)
        else:
            # Last resort: find the last number in the string
            numbers = re.findall(r"[-+]?\d*\.\d+|\d+", text)
            extracted = numbers[-1] if numbers else None

    if extracted:
        return extracted.replace(",", "")
    return None

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

---
## Part 3 - Baseline & CoT Prompting

In [4]:
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.bfloat16, device_map="auto"
)

def evaluate_on_test_set(model, dataset, use_cot=True, num_samples=None):
    model.eval()
    correct = 0
    total = num_samples if num_samples else len(dataset)

    print(f"Evaluating {total} samples...")
    for i in tqdm(range(total)):
        question = dataset[i]['prompt'][1]['content']
        gt_answer = dataset[i]['answer']

        if use_cot:
            prompt = f"Question: {question}\nLet's think step by step. End your response with 'The answer is X'."
        else:
            prompt = f"Question: {question}\nAnswer:"

        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=256, pad_token_id=tokenizer.eos_token_id)

        prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)
        extracted_ans = extract_numeric_answer(prediction)

        if extracted_ans == gt_answer:
            correct += 1

    accuracy = (correct / total) * 100
    return accuracy

# Run Baseline Evaluations
# Note: num_samples=100 for speed in lab; set to None for full test set
subset_size = 100
acc_baseline = evaluate_on_test_set(model, test_dataset, use_cot=False, num_samples=subset_size)
acc_cot = evaluate_on_test_set(model, test_dataset, use_cot=True, num_samples=subset_size)

print(f"\nBaseline Accuracy: {acc_baseline}%")
print(f"CoT Prompting Accuracy: {acc_cot}%")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Evaluating 100 samples...


  0%|          | 0/100 [00:06<?, ?it/s]
/pytorch/aten/src/ATen/native/cuda/TensorCompare.cu:109: _assert_async_cuda_kernel: block: [0,0,0], thread: [0,0,0] Assertion `probability tensor contains either `inf`, `nan` or element < 0` failed.


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


---
## Part 4 – GRPO Training Implementation

We define reward functions that the GRPOTrainer will use to calculate the advantages of the group.

In [ ]:
def extract_numeric_answer(text):
    """Robust extraction of the number after <answer> tags or as a fallback."""
    # Ensure text is a string
    if not isinstance(text, str):
        return None

    match = re.search(r"<answer>(.*?)</answer>", text, re.DOTALL)
    if match:
        extracted = match.group(1).strip()
    else:
        # Fallback for baseline/CoT
        ans_match = re.search(r"[Tt]he answer is\s*([\d\.,]+)", text)
        if ans_match:
            extracted = ans_match.group(1)
        else:
            numbers = re.findall(r"[-+]?\d*\.\d+|\d+", text)
            extracted = numbers[-1] if numbers else None

    return extracted.replace(",", "").strip() if extracted else None

def accuracy_reward_fn(prompts, completions, answer, **kwargs):
    rewards = []
    for completion, gt_answer in zip(completions, answer):
        # FIX: Extract content string from the list of messages
        # completion is usually [{'role': 'assistant', 'content': '...'}]
        content = completion[0]["content"] if isinstance(completion, list) else completion
        extracted = extract_numeric_answer(content)
        rewards.append(1.0 if extracted == gt_answer else 0.0)
    return rewards

def format_reward_fn(prompts, completions, **kwargs):
    rewards = []
    for completion in completions:
        # FIX: Extract content string
        content = completion[0]["content"] if isinstance(completion, list) else completion
        has_thought = "<thought>" in content and "</thought>" in content
        has_answer = "<answer>" in content and "</answer>" in content
        rewards.append(0.2 if (has_thought and has_answer) else 0.0)
    return rewards

# LoRA Configuration
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

# GRPO Config
training_args = GRPOConfig(
    output_dir="./grpo-results",
    learning_rate=1e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_generations=4, # Group size G
    max_completion_length=512,
    max_steps=100,
    save_steps=100,
    logging_steps=1,
    report_to="none"
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[accuracy_reward_fn, format_reward_fn],
    args=training_args,
    train_dataset=train_dataset,
    peft_config=peft_config,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Step,Training Loss
1,0.093333
2,-0.030704
3,-0.016847
4,0.054909
5,0.094881
6,-0.067492
7,0.007728
8,0.000000
9,0.000000
10,0.000000


TrainOutput(global_step=100, training_loss=0.014014031812548637, metrics={'train_runtime': 2985.6116, 'train_samples_per_second': 0.268, 'train_steps_per_second': 0.033, 'total_flos': 0.0, 'train_loss': 0.014014031812548637})

## Part 5 - Production-Scale RL with veRL

While `trl` is excellent for learning and small-scale LoRA, industrial-scale reasoning models (like DeepSeek-R1) use distributed frameworks.

### What is veRL?
**veRL (Volcano Engine Reinforcement Learning)** is an open-source framework by ByteDance designed for massive-scale RL training.

**Why use veRL for GRPO?**
1.  **Hybrid Engine:** It allows the Policy model to act as both an inference engine (during group sampling) and a training engine (during the weight update) efficiently on the same GPUs.
2.  **Model Sharding:** It supports FSDP (Fully Sharded Data Parallel) and Megatron-LM styles of parallelism, allowing you to train 70B+ parameter models.
3.  **Throughput:** In large clusters, veRL can be 10x faster than standard SFT wrappers by overlapping the generation of the group (rollout) with the backward pass (optimization).

**Usage Note:** In a multi-GPU environment, you would replace the TRL trainer with a `verl` YAML config and launch via `ray`.

---
## Part 6 – Evaluation & Comparison

In [ ]:
def evaluate_grpo_trained(model, dataset, num_samples=100):
    model.eval()
    correct = 0
    total = num_samples

    for i in tqdm(range(total)):
        # Apply the trained chat template logic
        prompt_msgs = dataset[i]['prompt']
        prompt = tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)

        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=512, pad_token_id=tokenizer.eos_token_id)

        prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)
        extracted_ans = extract_numeric_answer(prediction)

        if extracted_ans == dataset[i]['answer']:
            correct += 1

    return (correct / total) * 100

acc_grpo = evaluate_grpo_trained(model, test_dataset, num_samples=subset_size)

print("\n" + "="*30)
print("FINAL RESULTS COMPARISON")
print(f"Baseline (Direct):    {acc_baseline}%")
print(f"CoT Prompting:        {acc_cot}%")
print(f"GRPO RL-Trained:      {acc_grpo}%")
print("="*30)

100%|██████████| 100/100 [21:08<00:00, 12.68s/it]


FINAL RESULTS COMPARISON
Baseline (Direct):    2.0%
CoT Prompting:        5.0%
GRPO RL-Trained:      33.0%
